# **SISTEMA RAG MULTI-AGENTE CON TRANSFORMERS Y MISTRAL**
### **Dataset: Edmunds Consumer Car Ratings & Reviews (Kaggle)**

---

## **ARQUITECTURA DEL NOTEBOOK (4 AGENTES)**

| Agente | Rol | Responsabilidad |
|--------|-----|-----------------|
| **AGENTE 1** | Data Analyst | Define y fundamenta la elección del dataset |
| **AGENTE 2** | Data Engineer | Carga, analiza, normaliza y prepara los datos |
| **AGENTE 3** | NLP/ML Engineer | Aplica transformers (sentiment + embeddings), construye el RAG |
| **AGENTE 4** | AI Agent Architect | Implementa el agente autónomo con function calling |

---

# ==========================================
# AGENTE 1: DEFINICIÓN Y FUNDAMENTACIÓN DEL DATASET
# ==========================================

## **Dataset: Edmunds Consumer Car Ratings and Reviews**

### **Origen**
Kaggle — [ankkur13/edmundsconsumer-car-ratings-and-reviews](https://www.kaggle.com/datasets/ankkur13/edmundsconsumer-car-ratings-and-reviews)

### **¿Por qué este dataset?**
- **Texto enriquecido**: reseñas de consumidores reales sobre autos → ideal para embeddings semánticos y RAG
- **Datos estructurados**: rating numérico (1-5) → perfecto para el agente calculadora
- **62 marcas** incluyendo deportivas: Ferrari, Lamborghini, Porsche, McLaren, Aston Martin, Maserati, Lotus, etc.
- **Normalizable**: requiere limpieza de texto, estandarización de fechas, manejo de nulos y extracción de marca/modelo/año
- **Mediano-grande**: ~10K-15K reseñas por archivo, dataset completo de ~500K+ filas
- **Diferente del anterior**: antes trabajamos con ventas transaccionales, ahora con reseñas textuales de autos

### **Columnas del dataset**
| Columna | Tipo | Descripción |
|---------|------|-------------|
| `Review_Date` | Texto | Fecha de la reseña (requiere parseo) |
| `Author_Name` | Texto | Nombre del autor (muchos nulos) |
| `Vehicle_Title` | Texto | Título completo del vehículo (año, marca, modelo, trim) |
| `Review_Title` | Texto | Título/resumen de la reseña |
| `Review` | Texto | **Cuerpo completo de la reseña** → input principal para NLP |
| `Rating` | Float | Puntuación del consumidor (1.0 - 5.0) |

### **Estructura del dataset en Kaggle**
El dataset contiene archivos CSV separados por marca: `Scraped_Car_Review_{marca}.csv`

---

# ==========================================
# AGENTE 2: INGENIERÍA DE DATOS
# (Carga, Análisis Exploratorio y Normalización)
# ==========================================

### **FASE 2.1: INSTALACIÓN DE DEPENDENCIAS**

In [1]:
# Instalación silenciosa de todas las dependencias del ecosistema
!pip install -q kagglehub pandas numpy matplotlib seaborn
!pip install -q transformers sentence-transformers
!pip install -q langchain langchain-community langchain-mistralai faiss-cpu
!pip install -q mistralai

print("[ENTORNO] Todas las dependencias instaladas correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 856.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are insta

### **FASE 2.2: DESCARGA DEL DATASET DESDE KAGGLE**

Usamos `kagglehub` para descargar el dataset directamente. Esto requiere autenticación:
1. Ve a tu cuenta de Kaggle → Settings → Create API Token
2. Sube `kaggle.json` a Colab o configura las variables de entorno

In [2]:
import os
import kagglehub
import pandas as pd
import glob

# Configurar credenciales de Kaggle (requerido para kagglehub)
# En Colab puedes subir tu kaggle.json o configurar las variables:
# from google.colab import files
# files.upload()  # Subir kaggle.json
# os.makedirs('/root/.kaggle', exist_ok=True)
# !mv kaggle.json /root/.kaggle/
# !chmod 600 /root/.kaggle/kaggle.json

print("[KAGGLE] Descargando dataset: ankkur13/edmundsconsumer-car-ratings-and-reviews...")

try:
    path = kagglehub.dataset_download("ankkur13/edmundsconsumer-car-ratings-and-reviews")
    print(f"[KAGGLE] Dataset descargado en: {path}")
except Exception as e:
    print(f"[ERROR] No se pudo descargar desde Kaggle: {e}")
    print("[FALLBACK] Asegúrate de tener configurado kaggle.json o sube manualmente los CSVs.")
    path = None

[KAGGLE] Descargando dataset: ankkur13/edmundsconsumer-car-ratings-and-reviews...


100%|██████████| 50.3M/50.3M [00:02<00:00, 19.9MB/s]

Extracting files...


[KAGGLE] Dataset descargado en: /root/.cache/kagglehub/datasets/ankkur13/edmundsconsumer-car-ratings-and-reviews/versions/3


### **FASE 2.3: CARGA Y COMBINACIÓN DE ARCHIVOS**

Cargamos todos los archivos CSV (uno por marca) y los combinamos en un solo DataFrame añadiendo la columna `Brand`.

In [12]:
import os
import kagglehub
import pandas as pd
import glob

def cargar_todas_las_marcas(ruta_base):
    """
    Busca todos los archivos Scraped_Car_Review_*.csv en el directorio
    y los combina en un solo DataFrame, extrayendo la marca del nombre del archivo.
    """
    if ruta_base is None:
        print("[ERROR] No hay ruta de dataset disponible.")
        return None

    patron = os.path.join(ruta_base, "Scraped_Car_Review_*.csv")
    archivos = glob.glob(patron)

    print(f"[CARGA] Encontrados {len(archivos)} archivos CSV (una por marca).")

    if not archivos:
        print("[ERROR] No se encontraron archivos CSV en la ruta.")
        return None

    dataframes = []
    for i, archivo in enumerate(archivos):
        nombre_base = os.path.basename(archivo)
        # Extraer marca del nombre: 'Scraped_Car_Review_ferrari.csv' -> 'ferrari'
        marca = nombre_base.replace("Scraped_Car_Review_", "").replace(".csv", "")

        try:
            # Usar engine='python' para mayor robustez con CSVs complejos
            # Se elimina encoding='unicode_escape' ya que puede interferir con engine='python'
            df_temp = pd.read_csv(archivo, engine='python')
            df_temp['Brand'] = marca
            dataframes.append(df_temp)
        except Exception as e:
            print(f"  [AVISO] Error al leer {nombre_base}: {e}")

    df_completo = pd.concat(dataframes, ignore_index=True)
    print(f"[CARGA] Dataset combinado: {df_completo.shape[0]} filas x {df_completo.shape[1]} columnas")
    print(f"[CARGA] Marcas únicas cargadas: {df_completo['Brand'].nunique()}")

    return df_completo


df = cargar_todas_las_marcas(path)

if df is not None:
    print("\n[VISTA PREVIA] Primeras 3 filas:")
    display(df.head(3))

[CARGA] Encontrados 34 archivos CSV (una por marca).
[CARGA] Dataset combinado: 160597 filas x 8 columnas
[CARGA] Marcas únicas cargadas: 34

[VISTA PREVIA] Primeras 3 filas:


,Unnamed: 0,Review_Date,Author_Name,Vehicle_Title,Review_Title,Review,Rating,Brand
0,0,on 04/28/17 08:08 AM (PDT),Garrett Stites,2015 Ferrari 458 Italia Convertible Spider 2dr...,The best car around!,This car gets great gas mileage and is the be...,5.00,ferrari
1,1,on 11/19/11 16:47 PM (PST),debu99,2006 Ferrari 612 Scaglietti Coupe F1 2dr Coupe...,keeps on beeing just great,Owning the 612 now over 3 years and using it ...,4.75,ferrari
2,2,on 06/28/07 22:12 PM (PDT),Arnell Baylet,2006 Ferrari 612 Scaglietti Coupe F1 2dr Coupe...,"Incredible Ride, Sticker Shock, Low MPG",Best controllable acceleration ever witnessed...,5.00,ferrari


### **FASE 2.4: ANÁLISIS EXPLORATORIO (EDA)**

Diagnosticamos la estructura del dataset antes de normalizar.

In [4]:
def analisis_exploratorio(df):
    """
    Realiza un diagnóstico completo del dataset:
    - Shape, tipos de datos, nulos
    - Estadísticas descriptivas
    - Distribución de ratings y marcas
    - Métricas textuales para chunking
    """
    print("="*60)
    print("           ANÁLISIS EXPLORATORIO DEL DATASET")
    print("="*60)

    # 1. Estructura general
    print(f"\n[1] DIMENSIONES: {df.shape[0]} filas x {df.shape[1]} columnas\n")

    # 2. Mapeo de tipos y nulos
    info_cols = pd.DataFrame({
        'Tipo': df.dtypes,
        'No_Nulos': df.count(),
        'Nulos': df.isnull().sum(),
        '%_Nulos': (df.isnull().sum() / len(df)) * 100
    })
    print("[2] MAPEO DE COLUMNAS (tipos + nulos):")
    display(info_cols)

    # 3. Distribución de ratings
    print("\n[3] DISTRIBUCIÓN DE RATINGS:")
    print(df['Rating'].describe())

    # 4. Top 10 marcas con más reseñas
    print("\n[4] TOP 10 MARCAS CON MÁS RESEÑAS:")
    top_marcas = df['Brand'].value_counts().head(10)
    print(top_marcas)

    # 5. Métricas textuales (crítico para definir chunk size)
    df['_texto_completo'] = df.apply(
        lambda r: f"{str(r.get('Review_Title', ''))} {str(r.get('Review', ''))}", axis=1
    )
    longitudes = df['_texto_completo'].str.len()
    print(f"\n[5] MÉTRICAS TEXTUALES PARA CHUNKING:")
    print(f"  Longitud promedio (chars): {longitudes.mean():.1f}")
    print(f"  Longitud mediana (chars):  {longitudes.median():.1f}")
    print(f"  Tokens estimados (prom/4): {longitudes.mean() / 4:.1f}")
    print(f"  Reviews vacías: {(longitudes == 0).sum()}")

    # 6. Marcas deportivas disponibles
    marcas_deportivas = ['ferrari', 'lamborghini', 'porsche', 'mclaren',
                         'aston martin', 'maserati', 'lotus', 'bugatti']
    disponibles = [m for m in marcas_deportivas if m in df['Brand'].str.lower().unique()]
    print(f"\n[6] MARCAS DEPORTIVAS DETECTADAS: {disponibles}")

    return df


if df is not None:
    df = analisis_exploratorio(df)

           ANÁLISIS EXPLORATORIO DEL DATASET

[1] DIMENSIONES: 1711 filas x 8 columnas

[2] MAPEO DE COLUMNAS (tipos + nulos):


,Tipo,No_Nulos,Nulos,%_Nulos
Unnamed: 0,object,1710,1,0.058445
Review_Date,object,1640,71,4.149620
Author_Name,object,1614,97,5.669199
Vehicle_Title,object,1614,97,5.669199
Review_Title,object,1614,97,5.669199
Review,object,1614,97,5.669199
Rating,float64,1588,123,7.188778
Brand,object,1711,0,0.000000



[3] DISTRIBUCIÓN DE RATINGS:
count    1588.000000
mean        4.143105
std         1.257573
min         1.000000
25%         4.000000
50%         4.875000
75%         5.000000
max         5.000000
Name: Rating, dtype: float64

[4] TOP 10 MARCAS CON MÁS RESEÑAS:
Brand
ram            566
fiat           395
maserati       302
lotus          158
tesla          140
genesis         78
rolls-royce     39
maybach         32
mclaren          1
Name: count, dtype: int64

[5] MÉTRICAS TEXTUALES PARA CHUNKING:
  Longitud promedio (chars): 720.4
  Longitud mediana (chars):  567.0
  Tokens estimados (prom/4): 180.1
  Reviews vacías: 0

[6] MARCAS DEPORTIVAS DETECTADAS: ['mclaren', 'maserati', 'lotus']


### **FASE 2.5: NORMALIZACIÓN Y LIMPIEZA**

Aplicamos las transformaciones necesarias para dejar el dataset listo para NLP:

In [16]:
import pandas as pd
import re # Import regex module for more robust string operations

def normalizar_dataset(df):
    """
    Pipeline completo de normalización:
    1. Limpieza de espacios en encabezados
    2. Parseo de fechas
    3. Extracción de año, marca y modelo desde Vehicle_Title
    4. Limpieza de texto en reseñas
    5. Manejo de nulos
    6. Estandarización de categorías
    7. Filtrado de registros válidos
    """
    df_clean = df.copy()

    print("="*60)
    print("           NORMALIZACIÓN DEL DATASET")
    print("="*60)
    print(f"[INICIO] Dataset recibido para normalización: {df_clean.shape[0]} filas.") # Added print

    # 1. Estandarizar nombres de columnas
    df_clean.columns = df_clean.columns.str.strip().str.upper()
    print("\n[1] Encabezados estandarizados a MAYÚSCULAS.")

    # 2. Parseo de fechas: 'on 02/02/17 19:53 PM (PST)' -> datetime
    if 'REVIEW_DATE' in df_clean.columns:
        print("[2] Parseando fechas...")
        # Clean date string: remove 'on ' prefix, timezone in parentheses, and AM/PM indicators
        # Example: 'on 04/28/17 08:08 AM (PDT)' -> '04/28/17 08:08 AM'
        df_clean['REVIEW_DATE_TEMP'] = df_clean['REVIEW_DATE'].astype(str)
        # Remove 'on ' at the beginning
        df_clean['REVIEW_DATE_TEMP'] = df_clean['REVIEW_DATE_TEMP'].apply(lambda x: re.sub(r'^on\s*', '', x))
        # Remove timezone like '(PDT)'
        df_clean['REVIEW_DATE_TEMP'] = df_clean['REVIEW_DATE_TEMP'].apply(lambda x: re.sub(r'\s+\(\w{3}\)$', '', x))
        # Remove AM/PM indicators (case-insensitive) to standardize to 24-hour format
        df_clean['REVIEW_DATE_TEMP'] = df_clean['REVIEW_DATE_TEMP'].apply(lambda x: re.sub(r'\s*(AM|PM)', '', x, flags=re.IGNORECASE).strip())

        df_clean['REVIEW_DATE_PARSED'] = pd.to_datetime(
            df_clean['REVIEW_DATE_TEMP'], format='%m/%d/%y %H:%M', errors='coerce' # Changed format to 24-hour without AM/PM
        )
        # Extraer año de la reseña
        df_clean['REVIEW_YEAR'] = df_clean['REVIEW_DATE_PARSED'].dt.year

        # Print range only if valid dates exist
        if not df_clean['REVIEW_YEAR'].isnull().all():
             print(f"   Fechas parseadas correctamente. Rango: {int(df_clean['REVIEW_YEAR'].min())} - {int(df_clean['REVIEW_YEAR'].max())}")
        else:
             print("   Advertencia: No se pudieron parsear fechas válidas.")

        df_clean = df_clean.drop(columns=['REVIEW_DATE_TEMP']) # Clean up intermediate column

    # 3. Extraer Año, Marca y Modelo desde VEHICLE_TITLE
    #    Formato típico: "2004 Dodge Neon SRT-4 SRT-4 4dr Sedan (2.4L 4cyl Turbo 5M)"
    if 'VEHICLE_TITLE' in df_clean.columns:
        print("[3] Extrayendo año, marca y modelo desde VEHICLE_TITLE...")
        # Extraer año (primeros 4 dígitos)
        df_clean['CAR_YEAR'] = df_clean['VEHICLE_TITLE'].str.extract(r'(\b\d{4}\b)')[0].fillna('No especificado') # Ensure it's a series and fillna
        # Marca estandarizada desde la columna Brand
        if 'BRAND' in df_clean.columns:
            df_clean['CAR_MAKE'] = df_clean['BRAND'].str.strip().str.title()
        # Modelo: extraer después del año y la marca
        df_clean['CAR_MODEL'] = df_clean['VEHICLE_TITLE'].apply(
            lambda x: ' '.join(str(x).split()[2:4]) if isinstance(x, str) and len(str(x).split()) > 3 else 'No especificado'
        )

    # 4. Limpieza de texto en reseñas
    if 'REVIEW' in df_clean.columns:
        print("[4] Limpiando texto de reseñas...")
        df_clean['REVIEW'] = df_clean['REVIEW'].astype(str).str.strip()
        # Remover HTML tags si existen
        df_clean['REVIEW'] = df_clean['REVIEW'].str.replace(r'<[^>]+>', '', regex=True)
        # Remover espacios múltiples
        df_clean['REVIEW'] = df_clean['REVIEW'].str.replace(r'\s+', ' ', regex=True)
    if 'REVIEW_TITLE' in df_clean.columns:
        df_clean['REVIEW_TITLE'] = df_clean['REVIEW_TITLE'].astype(str).str.strip()

    # 5. Manejo de nulos
    print("[5] Manejando valores nulos...")
    # Columnas de texto: reemplazar nulos con 'No especificado'
    cols_texto = ['AUTHOR_NAME', 'REVIEW_TITLE', 'CAR_MODEL', 'CAR_YEAR'] # Added CAR_YEAR
    for col in cols_texto:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna('No especificado')
            df_clean[col] = df_clean[col].replace(
                ['nan', 'NaN', 'None', '', 'No especificado'], 'No especificado'
            )
    # Filtrar filas sin Review (no sirven para RAG)
    antes = len(df_clean)
    # Keep only rows where REVIEW is not empty or 'nan' string
    df_clean = df_clean[df_clean['REVIEW'].astype(str).str.strip() != '']
    df_clean = df_clean[df_clean['REVIEW'].astype(str) != 'nan']
    print(f"   Filas eliminadas por review vacía: {antes - len(df_clean)}")

    # 6. Estandarizar marcas a mayúsculas
    if 'CAR_MAKE' in df_clean.columns:
        df_clean['CAR_MAKE'] = df_clean['CAR_MAKE'].str.upper()

    print(f"\n[RESUMEN] Dataset normalizado: {len(df_clean)} filas")
    print(f"[RESUMEN] Nulos remanentes totales: {df_clean.isnull().sum().sum()}")

    return df_clean


if df is not None:
    print(f"[PRE-NORMALIZACION] df tiene {df.shape[0]} filas.") # Added print
    df_limpio = normalizar_dataset(df)

    # Guardar versión normalizada
    df_limpio.to_csv('autos_normalizado.csv', index=False)
    print("\n[SISTEMA] Dataset normalizado guardado como 'autos_normalizado.csv'.")
    display(df_limpio.head(2))

[PRE-NORMALIZACION] df tiene 160597 filas.
           NORMALIZACIÓN DEL DATASET
[INICIO] Dataset recibido para normalización: 160597 filas.

[1] Encabezados estandarizados a MAYÚSCULAS.
[2] Parseando fechas...
   Advertencia: No se pudieron parsear fechas válidas.
[3] Extrayendo año, marca y modelo desde VEHICLE_TITLE...
[4] Limpiando texto de reseñas...
[5] Manejando valores nulos...
   Filas eliminadas por review vacía: 591

[RESUMEN] Dataset normalizado: 160006 filas
[RESUMEN] Nulos remanentes totales: 419336

[SISTEMA] Dataset normalizado guardado como 'autos_normalizado.csv'.


,UNNAMED: 0,REVIEW_DATE,AUTHOR_NAME,VEHICLE_TITLE,REVIEW_TITLE,REVIEW,RATING,BRAND,REVIEW_DATE_PARSED,REVIEW_YEAR,CAR_YEAR,CAR_MAKE,CAR_MODEL
0,0,on 04/28/17 08:08 AM (PDT),Garrett Stites,2015 Ferrari 458 Italia Convertible Spider 2dr...,The best car around!,This car gets great gas mileage and is the bes...,5.00,ferrari,NaT,NaN,2015,FERRARI,458 Italia
1,1,on 11/19/11 16:47 PM (PST),debu99,2006 Ferrari 612 Scaglietti Coupe F1 2dr Coupe...,keeps on beeing just great,Owning the 612 now over 3 years and using it i...,4.75,ferrari,NaT,NaN,2006,FERRARI,612 Scaglietti


---
# ==========================================
# AGENTE 3: TRANSFORMERS Y SISTEMA RAG
# ==========================================

Este agente implementa:
1. **Clasificación de sentimiento** con `transformers` (HuggingFace pipeline)
2. **Embeddings semánticos** con `sentence-transformers`
3. **Índice vectorial** con FAISS
4. **Cadena RAG** con Mistral Large 3

### **FASE 3.1: CONFIGURACIÓN DEL ENTORNO Y API MISTRAL**

Configuramos la API key de Mistral (usar Secretos en Colab) y conectamos con Mistral Large 3.

In [6]:
import os

# En Google Colab, usa la pestaña de Secretos (icono de llave) para configurar MISTRAL_API_KEY
# O bien descomenta y pega tu API key directamente (no recomendado para producción):
# os.environ["MISTRAL_API_KEY"] = "tu-api-key-aqui"

try:
    from google.colab import userdata
    api_key = userdata.get('MISTRAL_API_KEY')
    os.environ["MISTRAL_API_KEY"] = api_key
    print("[COLAB] API Key cargada desde Secretos.")
except ImportError:
    print("[ENTORNO] No detectado Google Colab. Usando variable de entorno existente.")
except Exception as e:
    print(f"[AVISO] No se pudo cargar API Key: {e}")

from langchain_mistralai import ChatMistralAI

def inicializar_mistral():
    """
    Inicializa el modelo Mistral Large 3 con temperatura 0
    para respuestas deterministas y precisas.
    """
    try:
        llm = ChatMistralAI(model="mistral-large-latest", temperature=0.0)
        test = llm.invoke("Responde únicamente: CONEXIÓN OK")
        print(f"[MISTRAL] {test.content.strip()}")
        return llm
    except Exception as e:
        print(f"[ERROR] No se pudo conectar con Mistral: {e}")
        print("[SOLUCIÓN] Configura MISTRAL_API_KEY en Secretos (Colab) o como variable de entorno.")
        return None

llm = inicializar_mistral()

[COLAB] API Key cargada desde Secretos.
[MISTRAL] CONEXIÓN OK


### **FASE 3.2: ANÁLISIS DE SENTIMIENTO CON TRANSFORMERS**

Usamos el pipeline de `transformers` de HuggingFace para clasificar el sentimiento de cada reseña. Esto enriquece los documentos RAG con metadata emocional.

In [7]:
from transformers import pipeline
import numpy as np

# Cargar pipeline de sentiment analysis con DistilBERT (rápido en CPU)
# DistilBERT es una versión destilada de BERT que mantiene el 97% de precisión
# siendo 60% más rápida. Ideal para entornos con recursos limitados como Colab.
print("[TRANSFORMERS] Cargando pipeline de sentiment analysis (DistilBERT)...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512
)
print("[TRANSFORMERS] Pipeline listo.")


def analizar_sentimiento_lote(textos, batch_size=32):
    """
    Analiza el sentimiento de una lista de textos usando transformers.
    Retorna etiqueta (POSITIVE/NEGATIVE) y score de confianza.
    """
    resultados = sentiment_pipeline(textos, batch_size=batch_size)
    etiquetas = [r['label'] for r in resultados]
    scores = [r['score'] for r in resultados]
    return etiquetas, scores


# Tomamos una muestra representativa para el análisis de sentimiento
# (procesar 500K+ reseñas completas consumiría mucha memoria en Colab)
if df_limpio is not None:
    # Limitar a 1000 reseñas para mantener el notebook rápido en Colab
    muestra_size = min(1000, len(df_limpio))
    df_muestra = df_limpio.sample(n=muestra_size, random_state=42).copy()

    textos_review = df_muestra['REVIEW'].tolist()
    print(f"\n[TRANSFORMERS] Analizando sentimiento de {len(textos_review)} reseñas...")

    etiquetas, scores = analizar_sentimiento_lote(textos_review)
    df_muestra['SENTIMENT_LABEL'] = etiquetas
    df_muestra['SENTIMENT_SCORE'] = scores

    # Estadísticas de sentimiento
    print(f"\n[TRANSFORMERS] Distribución de sentimientos:")
    print(df_muestra['SENTIMENT_LABEL'].value_counts())
    print(f"\n[TRANSFORMERS] Score promedio POSITIVE: {df_muestra[df_muestra['SENTIMENT_LABEL'] == 'POSITIVE']['SENTIMENT_SCORE'].mean():.4f}")
    print(f"[TRANSFORMERS] Score promedio NEGATIVE: {df_muestra[df_muestra['SENTIMENT_LABEL'] == 'NEGATIVE']['SENTIMENT_SCORE'].mean():.4f}")

    # Mostrar ejemplos
    print("\n[EJEMPLOS] Reseñas con sentimiento POSITIVE (score alto):")
    display(df_muestra[df_muestra['SENTIMENT_LABEL'] == 'POSITIVE']\
            .sort_values('SENTIMENT_SCORE', ascending=False)[['REVIEW_TITLE', 'REVIEW', 'RATING', 'SENTIMENT_LABEL', 'SENTIMENT_SCORE']].head(2))
    print("\n[EJEMPLOS] Reseñas con sentimiento NEGATIVE (score alto):")
    display(df_muestra[df_muestra['SENTIMENT_LABEL'] == 'NEGATIVE']\
            .sort_values('SENTIMENT_SCORE', ascending=False)[['REVIEW_TITLE', 'REVIEW', 'RATING', 'SENTIMENT_LABEL', 'SENTIMENT_SCORE']].head(2))

[TRANSFORMERS] Cargando pipeline de sentiment analysis (DistilBERT)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[TRANSFORMERS] Pipeline listo.

[TRANSFORMERS] Analizando sentimiento de 1000 reseñas...

[TRANSFORMERS] Distribución de sentimientos:
SENTIMENT_LABEL
POSITIVE    610
NEGATIVE    390
Name: count, dtype: int64

[TRANSFORMERS] Score promedio POSITIVE: 0.9783
[TRANSFORMERS] Score promedio NEGATIVE: 0.9589

[EJEMPLOS] Reseñas con sentimiento POSITIVE (score alto):


,REVIEW_TITLE,REVIEW,RATING,SENTIMENT_LABEL,SENTIMENT_SCORE
987,Outstanding to drive and own a 2016 Maserati G...,Truly a wonderful experience to drive every time!,5.0,POSITIVE,0.999892
869,One of the best,This one of the most wonderful car,NaN,POSITIVE,0.999888



[EJEMPLOS] Reseñas con sentimiento NEGATIVE (score alto):


,REVIEW_TITLE,REVIEW,RATING,SENTIMENT_LABEL,SENTIMENT_SCORE
1056,Poor Quality,Im having the hardest time figuring out why I ...,3.5,NEGATIVE,0.999813
681,Lemon,Worst vehicle my company has ever purchased. I...,1.0,NEGATIVE,0.999812


### **FASE 3.3: CONSTRUCCIÓN DEL SISTEMA RAG**

Transformamos cada fila del dataset en un documento narrativo con metadata enriquecida (incluyendo sentimiento), generamos embeddings con `sentence-transformers`, indexamos con FAISS y ensamblamos la cadena RAG con Mistral.

In [8]:
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def construir_sistema_rag(df, llm):
    """
    Construye el pipeline RAG completo:
    1. Textualización semántica de cada fila
    2. Generación de embeddings con sentence-transformers
    3. Indexación FAISS
    4. Cadena RAG con LangChain Expression Language (LCEL)

    A diferencia de create_pandas_dataframe_agent que usa un prefix estático,
    aquí el contexto se recupera DINÁMICAMENTE desde FAISS, inyectándose
    automáticamente en la variable {context} del prompt.
    """
    documentos = []
    print("[RAG] Creando narrativas semánticas desde las reseñas...")

    for idx, fila in df.iterrows():
        # Construir narrativa con los datos más relevantes
        narrativa = (
            f"Reseña de auto - Marca: {fila.get('CAR_MAKE', 'N/E')}. "
            f"Modelo: {fila.get('CAR_MODEL', 'N/E')}. "
            f"Año del vehículo: {fila.get('CAR_YEAR', 'N/E')}. "
            f"Título de la reseña: {fila.get('REVIEW_TITLE', 'N/E')}. "
            f"Puntuación del consumidor: {fila.get('RATING', 'N/E')} de 5. "
            f"Sentimiento detectado: {fila.get('SENTIMENT_LABEL', 'N/E')} "
            f"(confianza: {fila.get('SENTIMENT_SCORE', 'N/E')}). "
            f"Texto completo de la reseña: {fila.get('REVIEW', 'N/E')}"
        )

        metadatos = {
            "marca": str(fila.get('CAR_MAKE', '')).upper(),
            "modelo": str(fila.get('CAR_MODEL', '')),
            "rating": float(fila.get('RATING', 0)) if pd.notna(fila.get('RATING')) else 0.0,
            "sentimiento": str(fila.get('SENTIMENT_LABEL', '')),
            "año_vehiculo": str(fila.get('CAR_YEAR', ''))
        }
        documentos.append(Document(page_content=narrativa, metadata=metadatos))

    # Generar embeddings usando Sentence Transformers (corre en CPU)
    print("[RAG] Generando embeddings con sentence-transformers/all-MiniLM-L6-v2...")
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    # Construir índice FAISS en memoria
    print("[RAG] Indexando documentos en FAISS...")
    vector_db = FAISS.from_documents(documentos, embeddings)
    retriever = vector_db.as_retriever(search_kwargs={"k": 4})
    print(f"[RAG] Índice FAISS creado con {len(documentos)} documentos.")

    # Prompt corporativo anti-alucinaciones
    system_prompt = (
        "Eres un Analista Senior de reseñas automotrices.\n"
        "Responde las preguntas del usuario basándote ESTRICTA y EXCLUSIVAMENTE "
        "en el contexto provisto abajo.\n"
        "Si el contexto no contiene los datos para responder, di textualmente: "
        "'Lo siento, la información disponible en el corpus analizado no contiene "
        "los datos específicos para responder esa pregunta.'\n\n"
        "CONTEXTO FACTUAL:\n{context}"
    )
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}")
    ])

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # Cadena RAG con LCEL moderna (reemplaza create_retrieval_chain)
    rag_chain = (
        {"context": retriever | format_docs, "input": RunnablePassthrough()}
        | prompt_template
        | llm
        | StrOutputParser()
    )

    # Wrapper para mantener compatibilidad con invoke({"input": ...})
    class RAGWrapper:
        def __init__(self, chain):
            self.chain = chain
        def invoke(self, inputs):
            res = self.chain.invoke(inputs["input"])
            return {"answer": res}

    return RAGWrapper(rag_chain)


# Construir el sistema RAG usando la muestra con sentimiento
if 'df_muestra' in locals() and llm is not None:
    pipeline_rag = construir_sistema_rag(df_muestra, llm)
    print("\n[ÉXITO] Sistema RAG construido y listo para consultas.")

    # Prueba de control
    consulta_prueba = "¿Qué reseñas existen para autos Ferrari con rating alto?"
    resultado = pipeline_rag.invoke({"input": consulta_prueba})
    print(f"\n[TEST RAG] Consulta: '{consulta_prueba}'")
    print(f"[TEST RAG] Respuesta:\n{resultado['answer']}")
else:
    print("[ERROR] Datos o LLM no disponibles. Ejecuta las celdas anteriores primero.")

/tmp/ipykernel_5670/4179543566.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


[RAG] Creando narrativas semánticas desde las reseñas...
[RAG] Generando embeddings con sentence-transformers/all-MiniLM-L6-v2...


/tmp/ipykernel_5670/4179543566.py:48: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[RAG] Indexando documentos en FAISS...
[RAG] Índice FAISS creado con 1000 documentos.

[ÉXITO] Sistema RAG construido y listo para consultas.

[TEST RAG] Consulta: '¿Qué reseñas existen para autos Ferrari con rating alto?'
[TEST RAG] Respuesta:
Lo siento, la información disponible en el corpus analizado no contiene los datos específicos para responder esa pregunta.


---
# ==========================================
# AGENTE 4: AGENTE AUTÓNOMO CON FUNCTION CALLING
# ==========================================

Implementamos un agente autónomo que usa Mistral Large 3 como motor de razonamiento. El agente puede:
- **Buscar en el RAG**: recuperar reseñas específicas por marca, modelo, sentimiento, etc.
- **Calcular estadísticas**: promedios de rating, conteos, operaciones matemáticas

El agente decide AUTÓNOMAMENTE qué herramienta usar según la pregunta del usuario (bucle ReAct).

In [9]:
from langchain_core.tools import Tool
from langchain_core.messages import HumanMessage, ToolMessage
import pandas as pd
import time

# =====================================================================
# HERRAMIENTA 1: Buscador RAG de reseñas de autos
# =====================================================================
def tool_buscar_resenas(query):
    """
    Busca en el índice vectorial FAISS las reseñas más relevantes
    según la consulta semántica del usuario.
    """
    if 'pipeline_rag' not in locals() and 'pipeline_rag' not in globals():
        return "Error: El sistema RAG no está inicializado."
    res = pipeline_rag.invoke({"input": query})
    return res["answer"]

# =====================================================================
# HERRAMIENTA 2: Calculadora/Python para análisis numérico
# =====================================================================
def tool_calculadora(expresion):
    """
    Evalúa expresiones matemáticas de forma segura.
    Útil para sumar ratings, promediar precios, etc.
    NOTA: Solo permite operaciones básicas (suma, resta, multiplicación, división).
    """
    try:
        # Entorno restringido por seguridad (no permite builtins peligrosos)
        res = eval(str(expresion), {"__builtins__": {}}, {})
        return f"Resultado del cálculo: {res}"
    except Exception as e:
        return f"No se pudo calcular. Error: {e}"

# =====================================================================
# HERRAMIENTA 3: Estadísticas del DataFrame
# =====================================================================
def tool_estadisticas_dataframe(consulta):
    """
    Ejecuta consultas de análisis sobre el DataFrame completo.
    Consultas soportadas: 'promedio rating {marca}', 'conteo {marca}', 'top marcas'
    """
    if 'df_limpio' not in dir() and 'df_limpio' not in globals():
        return "Error: DataFrame no disponible."

    consulta = str(consulta).lower()

    if 'promedio rating' in consulta:
        for marca in df_limpio['CAR_MAKE'].unique():
            if marca.lower() in consulta:
                prom = df_limpio[df_limpio['CAR_MAKE'] == marca]['RATING'].mean()
                count = df_limpio[df_limpio['CAR_MAKE'] == marca]['RATING'].count()
                return f"Rating promedio para {marca}: {prom:.2f} (basado en {count} reseñas)"
        return "Marca no encontrada. Especifica una marca válida."

    if 'conteo' in consulta or 'count' in consulta or 'cuántas' in consulta:
        for marca in df_limpio['CAR_MAKE'].unique():
            if marca.lower() in consulta:
                count = len(df_limpio[df_limpio['CAR_MAKE'] == marca])
                return f"Cantidad de reseñas para {marca}: {count}"
        return "Marca no encontrada."

    if 'top' in consulta or 'mejores' in consulta:
        top = df_limpio.groupby('CAR_MAKE')['RATING'].mean().sort_values(ascending=False).head(5)
        return f"Top 5 marcas mejor rating:\n{top.to_string()}"

    return "Consulta no reconocida. Prueba con: 'promedio rating Porsche', 'conteo Ferrari', 'top marcas'"


# Registrar las herramientas disponibles para el agente
herramientas = [
    Tool(
        name="Buscador_Resenas_RAG",
        func=tool_buscar_resenas,
        description="Busca reseñas de autos en el índice semántico. Útil para preguntas sobre reseñas específicas, "
                    "experiencias de usuarios, opiniones detalladas. Recibe una consulta en lenguaje natural."
    ),
    Tool(
        name="Calculadora",
        func=tool_calculadora,
        description="Realiza cálculos matemáticos exactos. Recibe una expresión como '4.5 + 3.2 + 5.0 / 3'. "
                    "Útil para promediar ratings, sumar valores, etc."
    ),
    Tool(
        name="Estadisticas_DataFrame",
        func=tool_estadisticas_dataframe,
        description="Obtiene estadísticas del dataset: promedio de rating por marca, conteo de reseñas, top marcas. "
                    "Recibe comandos como 'promedio rating Ferrari', 'conteo Porsche', 'top marcas'."
    )
]

print("[AGENTE] Herramientas registradas:")
for h in herramientas:
    print(f"  -> {h.name}: {h.description[:60]}...")

[AGENTE] Herramientas registradas:
  -> Buscador_Resenas_RAG: Busca reseñas de autos en el índice semántico. Útil para pre...
  -> Calculadora: Realiza cálculos matemáticos exactos. Recibe una expresión c...
  -> Estadisticas_DataFrame: Obtiene estadísticas del dataset: promedio de rating por mar...


In [15]:
def ejecutar_agente(pregunta, llm, tools):
    """
    Bucle principal del agente autónomo.

    Funcionamiento:
    1. El LLM recibe la pregunta del usuario
    2. Decide si necesita usar una herramienta (tool_call)
    3. Si llama a una herramienta, ejecuta la función y devuelve el resultado
    4. El LLM revisa el resultado y decide si necesita más herramientas
    5. Cuando tiene suficiente información, genera la respuesta final

    Límite de 5 iteraciones para evitar loops infinitos.
    """
    print("="*60)
    print("        INICIALIZANDO AGENTE AUTÓNOMO")
    print("="*60)

    dict_tools = {t.name: t.func for t in tools}
    llm_with_tools = llm.bind_tools(tools)

    mensajes = [
        HumanMessage(content=(
            "Eres un Agente Autónomo de Análisis Automotriz.\n"
            "Tu misión es responder preguntas sobre reseñas de autos usando las herramientas disponibles.\n"
            "Sé preciso, profesional y fundamenta tus respuestas en los datos.\n"
            "Si necesitas calcular promedios, usa la Calculadora.\n"
            "Si necesitas buscar reseñas, usa el Buscador_RAG.\n"
            "Si necesitas estadísticas del dataset, usa Estadisticas_DataFrame.\n\n"
            f"PREGUNTA DEL USUARIO: {pregunta}"
        ))
    ]

    for paso in range(5):
        # Pausa para evitar rate limiting (Error 429)
        time.sleep(10) # Aumentado el tiempo de espera para el LLM y herramientas a 10 segundos

        print(f"\n[PASO {paso + 1}] El agente está analizando...")
        respuesta = llm_with_tools.invoke(mensajes)
        mensajes.append(respuesta)

        if hasattr(respuesta, 'tool_calls') and respuesta.tool_calls:
            for tool_call in respuesta.tool_calls:
                nombre = tool_call['name']
                args = tool_call['args']
                tool_id = tool_call['id']

                print(f"  -> [HERRAMIENTA] {nombre}")

                if isinstance(args, dict) and args:
                    parametro = list(args.values())[0]
                else:
                    parametro = str(args)

                print(f"  -> [INPUT] {parametro[:100]}...")

                if nombre in dict_tools:
                    observacion = dict_tools[nombre](parametro)
                else:
                    observacion = f"Error: Herramienta '{nombre}' no disponible."

                print(f"  -> [OBSERVACIÓN] {str(observacion)[:150]}...")
                mensajes.append(ToolMessage(content=str(observacion), tool_call_id=tool_id))
        else:
            print("\n" + "="*60)
            print("           RESPUESTA FINAL DEL AGENTE")
            print("="*60)
            print(respuesta.content)
            print("="*60)
            return

    print("\n[LÍMITE] El agente alcanzó el máximo de pasos.")


# =====================================================================
# EJECUCIÓN DEL AGENTE
# =====================================================================
if llm is not None:
    if 'pipeline_rag' in locals() or 'pipeline_rag' in globals():
        print("\n" + "#"*60)
        print("#       EJEMPLO 1: CONSULTA COMBINADA (RAG + ESTADÍSTICAS)")
        print("#"*60)
        pregunta1 = "Dame un resumen de las reseñas de Porsche y calcula el rating promedio de la marca"
        ejecutar_agente(pregunta1, llm, herramientas)

        time.sleep(30) # Añadido un sleep adicional y aumentado el tiempo entre ejecuciones a 30 segundos

        print("\n" + "#"*60)
        print("#       EJEMPLO 2: CONSULTA COMPARATIVA")
        print("#"*60)
        pregunta2 = "¿Cuántas reseñas hay de Ferrari y cuál es su rating promedio? Compáralo con Lamborghini"
        ejecutar_agente(pregunta2, llm, herramientas)
    else:
        print("[ERROR] El pipeline RAG no está inicializado. Ejecuta la Fase 3.3 primero.")
else:
    print("[ERROR] Mistral no está configurado. Ejecuta la Fase 3.1 primero.")


############################################################
#       EJEMPLO 1: CONSULTA COMBINADA (RAG + ESTADÍSTICAS)
############################################################
        INICIALIZANDO AGENTE AUTÓNOMO

[PASO 1] El agente está analizando...
  -> [HERRAMIENTA] Estadisticas_DataFrame
  -> [INPUT] conteo reseñas Porsche...
  -> [OBSERVACIÓN] Marca no encontrada....
  -> [HERRAMIENTA] Estadisticas_DataFrame
  -> [INPUT] promedio rating Porsche...
  -> [OBSERVACIÓN] Marca no encontrada. Especifica una marca válida....
  -> [HERRAMIENTA] Buscador_Resenas_RAG
  -> [INPUT] reseñas de Porsche...
  -> [OBSERVACIÓN] Lo siento, la información disponible en el corpus analizado no contiene los datos específicos para responder esa pregunta....

[PASO 2] El agente está analizando...
  -> [HERRAMIENTA] Buscador_Resenas_RAG
  -> [INPUT] Porsche reseñas detalladas...
  -> [OBSERVACIÓN] Lo siento, la información disponible en el corpus analizado no contiene los datos específicos para res

---
# **CONCLUSIÓN**

## **Resumen de lo implementado**

| Componente | Tecnología | Propósito |
|------------|-----------|-----------|
| **Agente 1** | Análisis de dataset | Definir y justificar la elección del dataset |
| **Agente 2** | Pandas + EDA | Carga, limpieza y normalización de datos |
| **Agente 3 - Transformers** | DistilBERT (HuggingFace) | Análisis de sentimiento de reseñas |
| **Agente 3 - RAG** | sentence-transformers + FAISS + Mistral | Búsqueda semántica y generación aumentada |
| **Agente 4** | LangChain + Function Calling | Agente autónomo con razonamiento y herramientas |

## **Próximos pasos**
- Aumentar la muestra de reseñas para mejorar la cobertura del RAG
- Probar con otros modelos de transformers (RoBERTa, DeBERTa)
- Agregar más herramientas al agente (gráficos, clustering, etc.)
- Desplegar como aplicación web con Gradio o Streamlit

---